In [4]:
# ============================================================
# 05 - FTL vs Carting Recommendation and Business Impact
# ============================================================

# -----------------------------
# Basic utilities
# -----------------------------
import os
import warnings

warnings.filterwarnings("ignore")

# -----------------------------
# Data handling
# -----------------------------
import numpy as np
import pandas as pd

# -----------------------------
# Visualization
# -----------------------------
import matplotlib.pyplot as plt

# -----------------------------
# Preprocessing
# -----------------------------
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# -----------------------------
# ML models
# -----------------------------
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

# -----------------------------
# Evaluation
# -----------------------------
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# -----------------------------
# Saving/loading models if needed
# -----------------------------
import joblib

# -----------------------------
# Display settings
# -----------------------------
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

# -----------------------------
# Folder setup
# -----------------------------
os.makedirs("outputs/tables", exist_ok=True)
os.makedirs("outputs/figures", exist_ok=True)

print("Imports loaded successfully.")

Imports loaded successfully.


In [5]:
# ============================================================
# Load required processed files
# ============================================================

trip_df = pd.read_csv("trip_corridor_cleaned.csv")
corridor_summary = pd.read_csv("corridor_summary.csv")
node_features = pd.read_csv("node_features_bottleneck.csv")

print("trip_df shape:", trip_df.shape)
print("corridor_summary shape:", corridor_summary.shape)
print("node_features shape:", node_features.shape)

best_final_model = joblib.load("models/final_graph_xgboost_ablation_model.pkl")

print("Final model loaded successfully.")
display(trip_df.head())

trip_df shape: (26368, 29)
corridor_summary shape: (2783, 14)
node_features shape: (1657, 24)
Final model loaded successfully.


,data,trip_creation_time,route_schedule_uuid,route_type,trip_uuid,source_center,source_name,destination_center,destination_name,od_start_time,od_end_time,start_scan_to_end_scan,is_cutoff,cutoff_factor,cutoff_timestamp,actual_distance_to_destination,actual_time,osrm_time,osrm_distance,factor,segment_actual_time,segment_osrm_time,segment_osrm_distance,segment_factor,delay_ratio,is_delayed,corridor,calculated_factor,delay_label_check
0,training,2018-09-12 00:00:16.535741,thanos::sroute:d7c989ba-a29b-4a0b-b2f4-288cdc6...,FTL,trip-153671041653548748,IND209304AAA,Kanpur_Central_H_6 (Uttar Pradesh),IND000000ACB,Gurgaon_Bilaspur_HB (Haryana),2018-09-12 16:39:46.858469,2018-09-13 13:40:23.123744,1260.0000,False,383,2018-09-13 01:18:26,383.7592,732.0000,329.0000,446.5496,2.2249,20.0000,10.0000,15.0693,2.0000,2.2249,1,IND209304AAA -> IND000000ACB,2.2249,1
1,training,2018-09-12 00:00:16.535741,thanos::sroute:d7c989ba-a29b-4a0b-b2f4-288cdc6...,FTL,trip-153671041653548748,IND462022AAA,Bhopal_Trnsport_H (Madhya Pradesh),IND209304AAA,Kanpur_Central_H_6 (Uttar Pradesh),2018-09-12 00:00:16.535741,2018-09-12 16:39:46.858469,999.0000,False,440,2018-09-12 01:50:28,440.9737,830.0000,388.0000,544.8027,2.1392,22.0000,3.0000,5.3898,7.3333,2.1392,1,IND462022AAA -> IND209304AAA,2.1392,1
2,training,2018-09-12 00:00:22.886430,thanos::sroute:3a1b0ab2-bb0b-4c53-8c59-eb2a2c0...,Carting,trip-153671042288605164,IND561203AAB,Doddablpur_ChikaDPP_D (Karnataka),IND562101AAA,Chikblapur_ShntiSgr_D (Karnataka),2018-09-12 02:03:09.655591,2018-09-12 03:01:59.598855,58.0000,False,24,2018-09-12 02:11:39.065776,24.6440,47.0000,26.0000,28.1994,1.8077,15.0000,7.0000,6.9464,2.1429,1.8077,1,IND561203AAB -> IND562101AAA,1.8077,1
3,training,2018-09-12 00:00:22.886430,thanos::sroute:3a1b0ab2-bb0b-4c53-8c59-eb2a2c0...,Carting,trip-153671042288605164,IND572101AAA,Tumkur_Veersagr_I (Karnataka),IND561203AAB,Doddablpur_ChikaDPP_D (Karnataka),2018-09-12 00:00:22.886430,2018-09-12 02:03:09.655591,122.0000,False,48,2018-09-12 00:17:19,48.5429,96.0000,42.0000,56.9116,2.2857,20.0000,3.0000,3.8074,6.6667,2.2857,1,IND572101AAA -> IND561203AAB,2.2857,1
4,training,2018-09-12 00:00:33.691250,thanos::sroute:de5e208e-7641-45e6-8100-4d9fb1e...,FTL,trip-153671043369099517,IND000000ACB,Gurgaon_Bilaspur_HB (Haryana),IND160002AAC,Chandigarh_Mehmdpur_H (Punjab),2018-09-14 03:40:17.106733,2018-09-14 17:34:55.442454,834.0000,False,237,2018-09-14 07:18:47,237.4396,611.0000,212.0000,281.2109,2.8821,275.0000,28.0000,32.8506,9.8214,2.8821,1,IND000000ACB -> IND160002AAC,2.8821,1


In [6]:
graph_test_enriched = pd.read_csv("graph_test_enriched.csv")
X_test_ablation = pd.read_csv("X_test_ablation.csv")

In [10]:
decision_df = graph_test_enriched.copy()

decision_df["predicted_actual_time"] = best_final_model.predict(X_test_ablation)

decision_df["prediction_error"] = decision_df["actual_time"] - decision_df["predicted_actual_time"]
decision_df["absolute_error"] = abs(decision_df["prediction_error"])

display(decision_df[
    [
        "source_center",
        "destination_center",
        "corridor",
        "route_type",
        "actual_time",
        "predicted_actual_time",
        "osrm_time",
        "actual_distance_to_destination"
    ]
].head())

,source_center,destination_center,corridor,route_type,actual_time,predicted_actual_time,osrm_time,actual_distance_to_destination
0,IND641664AAA,IND638657AAA,IND641664AAA -> IND638657AAA,Carting,51.0000,56.5261,37.0000,36.3180
1,IND638657AAA,IND638701AAA,IND638657AAA -> IND638701AAA,Carting,44.0000,50.0786,32.0000,31.4684
2,IND638701AAA,IND641607AAA,IND638701AAA -> IND641607AAA,Carting,39.0000,46.5932,21.0000,21.1376
3,IND641607AAA,IND641654AAA,IND641607AAA -> IND641654AAA,Carting,36.0000,33.5019,16.0000,15.2426
4,IND641654AAA,IND641664AAA,IND641654AAA -> IND641664AAA,Carting,33.0000,39.8125,26.0000,20.6848


In [11]:
from sklearn.preprocessing import MinMaxScaler

risk_cols = [
    "corridor_median_delay_ratio",
    "corridor_delay_rate",
    "corridor_train_trips",
    "source_bottleneck_score",
    "destination_bottleneck_score",
    "actual_distance_to_destination"
]

risk_data = decision_df[risk_cols].copy()

risk_data = risk_data.fillna(0)

scaler = MinMaxScaler()
risk_scaled = scaler.fit_transform(risk_data)

risk_scaled_df = pd.DataFrame(
    risk_scaled,
    columns=[col + "_risk_scaled" for col in risk_cols],
    index=decision_df.index
)

decision_df = pd.concat([decision_df, risk_scaled_df], axis=1)

decision_df["delay_risk_score"] = (
    0.25 * decision_df["corridor_median_delay_ratio_risk_scaled"] +
    0.25 * decision_df["corridor_delay_rate_risk_scaled"] +
    0.15 * decision_df["corridor_train_trips_risk_scaled"] +
    0.15 * decision_df["source_bottleneck_score_risk_scaled"] +
    0.15 * decision_df["destination_bottleneck_score_risk_scaled"] +
    0.05 * decision_df["actual_distance_to_destination_risk_scaled"]
)

display(decision_df[[
    "corridor",
    "route_type",
    "predicted_actual_time",
    "delay_risk_score",
    "corridor_median_delay_ratio",
    "corridor_delay_rate",
    "source_bottleneck_score",
    "destination_bottleneck_score"
]].head())

,corridor,route_type,predicted_actual_time,delay_risk_score,corridor_median_delay_ratio,corridor_delay_rate,source_bottleneck_score,destination_bottleneck_score
0,IND641664AAA -> IND638657AAA,Carting,56.5261,0.3218,1.4474,1.0000,0.1603,0.1609
1,IND638657AAA -> IND638701AAA,Carting,50.0786,0.3234,1.4747,1.0000,0.1609,0.1617
2,IND638701AAA -> IND641607AAA,Carting,46.5932,0.3311,2.2273,1.0000,0.1617,0.1768
3,IND641607AAA -> IND641654AAA,Carting,33.5019,0.3224,2.0000,1.0000,0.1768,0.1570
4,IND641654AAA -> IND641664AAA,Carting,39.8125,0.3173,1.5200,1.0000,0.1570,0.1603


In [12]:
def risk_category(score):
    if score >= 0.70:
        return "High"
    elif score >= 0.40:
        return "Medium"
    else:
        return "Low"

decision_df["delay_risk_category"] = decision_df["delay_risk_score"].apply(risk_category)

display(decision_df["delay_risk_category"].value_counts())
display((decision_df["delay_risk_category"].value_counts(normalize=True) * 100).round(2))

delay_risk_category
Low       4249
Medium    1025
Name: count, dtype: int64

delay_risk_category
Low      80.5700
Medium   19.4300
Name: proportion, dtype: float64

In [13]:
distance_threshold = decision_df["actual_distance_to_destination"].median()
print("Distance threshold:", distance_threshold)

Distance threshold: 33.0774333969571


In [14]:
def recommend_route_type(row):
    risk = row["delay_risk_category"]
    distance = row["actual_distance_to_destination"]

    if risk == "High":
        return "FTL"
    elif risk == "Medium" and distance >= distance_threshold:
        return "FTL"
    else:
        return "Carting"

decision_df["recommended_route_type"] = decision_df.apply(recommend_route_type, axis=1)

display(decision_df["recommended_route_type"].value_counts())

recommended_route_type
Carting    4661
FTL         613
Name: count, dtype: int64

In [15]:
def recommendation_reason(row):
    risk = row["delay_risk_category"]
    distance = row["actual_distance_to_destination"]
    source_score = row["source_bottleneck_score"]
    dest_score = row["destination_bottleneck_score"]
    corridor_delay = row["corridor_median_delay_ratio"]

    if row["recommended_route_type"] == "FTL":
        if risk == "High":
            return (
                "Recommended FTL because corridor delay risk is high, "
                "with elevated historical delay and/or bottleneck hub exposure."
            )
        else:
            return (
                "Recommended FTL because risk is medium and distance is above the median, "
                "so a more reliable movement option is preferred."
            )
    else:
        return (
            "Recommended Carting because delay risk is low or distance is shorter, "
            "so cost-efficient movement may be acceptable."
        )

decision_df["recommendation_reason"] = decision_df.apply(recommendation_reason, axis=1)

display(decision_df[
    [
        "corridor",
        "route_type",
        "recommended_route_type",
        "delay_risk_category",
        "delay_risk_score",
        "predicted_actual_time",
        "recommendation_reason"
    ]
].head(10))

,corridor,route_type,recommended_route_type,delay_risk_category,delay_risk_score,predicted_actual_time,recommendation_reason
0,IND641664AAA -> IND638657AAA,Carting,Carting,Low,0.3218,56.5261,Recommended Carting because delay risk is low ...
1,IND638657AAA -> IND638701AAA,Carting,Carting,Low,0.3234,50.0786,Recommended Carting because delay risk is low ...
2,IND638701AAA -> IND641607AAA,Carting,Carting,Low,0.3311,46.5932,Recommended Carting because delay risk is low ...
3,IND641607AAA -> IND641654AAA,Carting,Carting,Low,0.3224,33.5019,Recommended Carting because delay risk is low ...
4,IND641654AAA -> IND641664AAA,Carting,Carting,Low,0.3173,39.8125,Recommended Carting because delay risk is low ...
5,IND560067AAC -> IND562114AAA,Carting,Carting,Low,0.3377,29.2026,Recommended Carting because delay risk is low ...
6,IND209101AAA -> IND209111AAA,FTL,Carting,Low,0.1706,51.5252,Recommended Carting because delay risk is low ...
7,IND209304AAA -> IND209101AAA,FTL,Carting,Low,0.3016,64.2142,Recommended Carting because delay risk is low ...
8,IND285001AAD -> IND285205AAB,FTL,Carting,Low,0.3088,46.0552,Recommended Carting because delay risk is low ...
9,IND209304AAA -> IND209206AAB,Carting,Carting,Low,0.1002,52.0962,Recommended Carting because delay risk is low ...


In [16]:
decision_df["route_type_changed"] = (
    decision_df["route_type"] != decision_df["recommended_route_type"]
)

print("Recommendation change count:")
display(decision_df["route_type_changed"].value_counts())

print("\nRecommendation change percentage:")
display((decision_df["route_type_changed"].value_counts(normalize=True) * 100).round(2))

Recommendation change count:


route_type_changed
False    2775
True     2499
Name: count, dtype: int64


Recommendation change percentage:


route_type_changed
False   52.6200
True    47.3800
Name: proportion, dtype: float64

In [17]:
recommendation_summary = (
    decision_df
    .groupby(["delay_risk_category", "recommended_route_type"])
    .agg(
        records=("predicted_actual_time", "count"),
        median_predicted_time=("predicted_actual_time", "median"),
        median_distance=("actual_distance_to_destination", "median"),
        median_risk_score=("delay_risk_score", "median")
    )
    .reset_index()
    .sort_values(["delay_risk_category", "recommended_route_type"])
)

display(recommendation_summary)

,delay_risk_category,recommended_route_type,records,median_predicted_time,median_distance,median_risk_score
0,Low,Carting,4249,71.5550,31.4684,0.3277
1,Medium,Carting,412,76.9113,21.7881,0.4381
2,Medium,FTL,613,315.6459,99.8836,0.4714


In [18]:
decision_output_cols = [
    "source_center",
    "destination_center",
    "corridor",
    "route_type",
    "recommended_route_type",
    "route_type_changed",
    "actual_time",
    "predicted_actual_time",
    "osrm_time",
    "actual_distance_to_destination",
    "delay_risk_score",
    "delay_risk_category",
    "corridor_median_delay_ratio",
    "corridor_delay_rate",
    "source_bottleneck_score",
    "destination_bottleneck_score",
    "recommendation_reason"
]

# Add trip_uuid only if it exists
if "trip_uuid" in decision_df.columns:
    decision_output_cols = ["trip_uuid"] + decision_output_cols

decision_df[decision_output_cols].to_csv(
    "outputs/tables/ftl_carting_recommendations.csv",
    index=False
)

recommendation_summary.to_csv(
    "outputs/tables/ftl_carting_recommendation_summary.csv",
    index=False
)

print("Saved FTL vs Carting recommendation outputs.")

Saved FTL vs Carting recommendation outputs.


# Important Improvement Before Final Report

In [19]:
low_cutoff = decision_df["delay_risk_score"].quantile(0.50)
high_cutoff = decision_df["delay_risk_score"].quantile(0.80)

print("Low cutoff:", low_cutoff)
print("High cutoff:", high_cutoff)

def risk_category_quantile(score):
    if score >= high_cutoff:
        return "High"
    elif score >= low_cutoff:
        return "Medium"
    else:
        return "Low"

decision_df["delay_risk_category_v2"] = decision_df["delay_risk_score"].apply(risk_category_quantile)

display(decision_df["delay_risk_category_v2"].value_counts())
display((decision_df["delay_risk_category_v2"].value_counts(normalize=True) * 100).round(2))

Low cutoff: 0.3328101469688358
High cutoff: 0.3984049502098059


delay_risk_category_v2
Low       2637
Medium    1582
High      1055
Name: count, dtype: int64

delay_risk_category_v2
Low      50.0000
Medium   30.0000
High     20.0000
Name: proportion, dtype: float64

In [20]:
distance_threshold = decision_df["actual_distance_to_destination"].median()

def recommend_route_type_v2(row):
    risk = row["delay_risk_category_v2"]
    distance = row["actual_distance_to_destination"]

    if risk == "High":
        return "FTL"
    elif risk == "Medium" and distance >= distance_threshold:
        return "FTL"
    else:
        return "Carting"

decision_df["recommended_route_type_v2"] = decision_df.apply(recommend_route_type_v2, axis=1)

display(decision_df["recommended_route_type_v2"].value_counts())

recommended_route_type_v2
Carting    3425
FTL        1849
Name: count, dtype: int64

In [21]:
recommendation_summary_v2 = (
    decision_df
    .groupby(["delay_risk_category_v2", "recommended_route_type_v2"])
    .agg(
        records=("predicted_actual_time", "count"),
        median_predicted_time=("predicted_actual_time", "median"),
        median_distance=("actual_distance_to_destination", "median"),
        median_risk_score=("delay_risk_score", "median")
    )
    .reset_index()
    .sort_values(["delay_risk_category_v2", "recommended_route_type_v2"])
)

display(recommendation_summary_v2)

,delay_risk_category_v2,recommended_route_type_v2,records,median_predicted_time,median_distance,median_risk_score
0,High,FTL,1055,119.1373,38.7165,0.4616
1,Low,Carting,2637,62.3688,31.3472,0.3184
2,Medium,Carting,788,58.2268,15.9927,0.3523
3,Medium,FTL,794,194.6231,78.4579,0.3507


In [22]:
decision_df["route_type_changed_v2"] = (
    decision_df["route_type"] != decision_df["recommended_route_type_v2"]
)

decision_output_cols_v2 = [
    "source_center",
    "destination_center",
    "corridor",
    "route_type",
    "recommended_route_type_v2",
    "route_type_changed_v2",
    "actual_time",
    "predicted_actual_time",
    "osrm_time",
    "actual_distance_to_destination",
    "delay_risk_score",
    "delay_risk_category_v2",
    "corridor_median_delay_ratio",
    "corridor_delay_rate",
    "source_bottleneck_score",
    "destination_bottleneck_score"
]

if "trip_uuid" in decision_df.columns:
    decision_output_cols_v2 = ["trip_uuid"] + decision_output_cols_v2

decision_df[decision_output_cols_v2].to_csv(
    "outputs/tables/ftl_carting_recommendations_v2.csv",
    index=False
)

recommendation_summary_v2.to_csv(
    "outputs/tables/ftl_carting_recommendation_summary_v2.csv",
    index=False
)

print("Saved V2 FTL vs Carting recommendation outputs.")

Saved V2 FTL vs Carting recommendation outputs.


# Logic

In [23]:
impact_df = decision_df.copy()

print("Impact data shape:", impact_df.shape)

display(impact_df[
    [
        "corridor",
        "route_type",
        "recommended_route_type_v2",
        "actual_time",
        "predicted_actual_time",
        "osrm_time",
        "delay_risk_score",
        "delay_risk_category_v2"
    ]
].head())

Impact data shape: (5274, 57)


,corridor,route_type,recommended_route_type_v2,actual_time,predicted_actual_time,osrm_time,delay_risk_score,delay_risk_category_v2
0,IND641664AAA -> IND638657AAA,Carting,Carting,51.0000,56.5261,37.0000,0.3218,Low
1,IND638657AAA -> IND638701AAA,Carting,Carting,44.0000,50.0786,32.0000,0.3234,Low
2,IND638701AAA -> IND641607AAA,Carting,Carting,39.0000,46.5932,21.0000,0.3311,Low
3,IND641607AAA -> IND641654AAA,Carting,Carting,36.0000,33.5019,16.0000,0.3224,Low
4,IND641654AAA -> IND641664AAA,Carting,Carting,33.0000,39.8125,26.0000,0.3173,Low


In [24]:
impact_df["actual_sla_breach"] = (
    impact_df["actual_time"] > 1.2 * impact_df["osrm_time"]
).astype(int)

impact_df["predicted_sla_breach"] = (
    impact_df["predicted_actual_time"] > 1.2 * impact_df["osrm_time"]
).astype(int)

print("Actual SLA breach distribution:")
display(impact_df["actual_sla_breach"].value_counts())
display((impact_df["actual_sla_breach"].value_counts(normalize=True) * 100).round(2))

print("\nPredicted SLA breach distribution:")
display(impact_df["predicted_sla_breach"].value_counts())
display((impact_df["predicted_sla_breach"].value_counts(normalize=True) * 100).round(2))

Actual SLA breach distribution:


actual_sla_breach
1    4958
0     316
Name: count, dtype: int64

actual_sla_breach
1   94.0100
0    5.9900
Name: proportion, dtype: float64


Predicted SLA breach distribution:


predicted_sla_breach
1    5004
0     270
Name: count, dtype: int64

predicted_sla_breach
1   94.8800
0    5.1200
Name: proportion, dtype: float64

In [25]:
impact_df["actual_delay_time"] = impact_df["actual_time"] - impact_df["osrm_time"]
impact_df["predicted_delay_time"] = impact_df["predicted_actual_time"] - impact_df["osrm_time"]

# Delay cannot be negative for business impact estimation
impact_df["actual_delay_time_positive"] = impact_df["actual_delay_time"].clip(lower=0)
impact_df["predicted_delay_time_positive"] = impact_df["predicted_delay_time"].clip(lower=0)

print("Total actual delay time:", impact_df["actual_delay_time_positive"].sum())
print("Median actual delay time:", impact_df["actual_delay_time_positive"].median())
print("Mean actual delay time:", impact_df["actual_delay_time_positive"].mean())

Total actual delay time: 563814.0
Median actual delay time: 38.0
Mean actual delay time: 106.90443686006826


In [26]:
impact_df["action_required"] = (
    (impact_df["delay_risk_category_v2"] == "High") |
    (impact_df["route_type_changed_v2"] == True)
).astype(int)

action_df = impact_df[impact_df["action_required"] == 1].copy()

print("Total records:", len(impact_df))
print("Action-required records:", len(action_df))
print("Action-required percentage:", round(len(action_df) / len(impact_df) * 100, 2))

print("\nActual delay time in action-required records:")
print(action_df["actual_delay_time_positive"].sum())

Total records: 5274
Action-required records: 2834
Action-required percentage: 53.74

Actual delay time in action-required records:
379550.0


In [27]:
total_actual_delay = impact_df["actual_delay_time_positive"].sum()
action_delay = action_df["actual_delay_time_positive"].sum()
actual_sla_breaches = impact_df["actual_sla_breach"].sum()
action_sla_breaches = action_df["actual_sla_breach"].sum()

scenarios = {
    "Conservative": 0.05,
    "Moderate": 0.10,
    "Optimistic": 0.15
}

impact_results = []

for scenario_name, reduction_rate in scenarios.items():
    estimated_delay_time_saved = action_delay * reduction_rate

    estimated_sla_breaches_reduced = action_sla_breaches * reduction_rate

    impact_results.append({
        "scenario": scenario_name,
        "assumed_reduction_rate": reduction_rate,
        "action_required_records": len(action_df),
        "action_delay_time": action_delay,
        "estimated_delay_time_saved": estimated_delay_time_saved,
        "actual_sla_breaches": actual_sla_breaches,
        "action_sla_breaches": action_sla_breaches,
        "estimated_sla_breaches_reduced": estimated_sla_breaches_reduced
    })

impact_summary = pd.DataFrame(impact_results)

display(impact_summary)

,scenario,assumed_reduction_rate,action_required_records,action_delay_time,estimated_delay_time_saved,actual_sla_breaches,action_sla_breaches,estimated_sla_breaches_reduced
0,Conservative,0.0500,2834,379550.0000,18977.5000,4958,2657,132.8500
1,Moderate,0.1000,2834,379550.0000,37955.0000,4958,2657,265.7000
2,Optimistic,0.1500,2834,379550.0000,56932.5000,4958,2657,398.5500


In [28]:
penalty_per_sla_breach = 100  # assumption, change if needed

impact_summary["estimated_penalty_saved"] = (
    impact_summary["estimated_sla_breaches_reduced"] * penalty_per_sla_breach
)

display(impact_summary)

,scenario,assumed_reduction_rate,action_required_records,action_delay_time,estimated_delay_time_saved,actual_sla_breaches,action_sla_breaches,estimated_sla_breaches_reduced,estimated_penalty_saved
0,Conservative,0.0500,2834,379550.0000,18977.5000,4958,2657,132.8500,13285.0000
1,Moderate,0.1000,2834,379550.0000,37955.0000,4958,2657,265.7000,26570.0000
2,Optimistic,0.1500,2834,379550.0000,56932.5000,4958,2657,398.5500,39855.0000


In [29]:
risk_impact_summary = (
    impact_df
    .groupby("delay_risk_category_v2")
    .agg(
        records=("predicted_actual_time", "count"),
        actual_sla_breaches=("actual_sla_breach", "sum"),
        sla_breach_rate=("actual_sla_breach", "mean"),
        total_actual_delay_time=("actual_delay_time_positive", "sum"),
        median_actual_delay_time=("actual_delay_time_positive", "median"),
        median_predicted_time=("predicted_actual_time", "median"),
        median_distance=("actual_distance_to_destination", "median")
    )
    .reset_index()
)

risk_impact_summary["sla_breach_rate_pct"] = (
    risk_impact_summary["sla_breach_rate"] * 100
).round(2)

display(risk_impact_summary)

,delay_risk_category_v2,records,actual_sla_breaches,sla_breach_rate,total_actual_delay_time,median_actual_delay_time,median_predicted_time,median_distance,sla_breach_rate_pct
0,High,1055,1029,0.9754,267804.0000,66.0000,119.1373,38.7165,97.5400
1,Low,2637,2366,0.8972,139053.0000,28.0000,62.3688,31.3472,89.7200
2,Medium,1582,1563,0.9880,156957.0000,51.0000,98.0095,33.6002,98.8000


In [30]:
recommendation_impact_summary = (
    impact_df
    .groupby("recommended_route_type_v2")
    .agg(
        records=("predicted_actual_time", "count"),
        actual_sla_breaches=("actual_sla_breach", "sum"),
        sla_breach_rate=("actual_sla_breach", "mean"),
        total_actual_delay_time=("actual_delay_time_positive", "sum"),
        median_actual_delay_time=("actual_delay_time_positive", "median"),
        median_predicted_time=("predicted_actual_time", "median"),
        median_distance=("actual_distance_to_destination", "median")
    )
    .reset_index()
)

recommendation_impact_summary["sla_breach_rate_pct"] = (
    recommendation_impact_summary["sla_breach_rate"] * 100
).round(2)

display(recommendation_impact_summary)

,recommended_route_type_v2,records,actual_sla_breaches,sla_breach_rate,total_actual_delay_time,median_actual_delay_time,median_predicted_time,median_distance,sla_breach_rate_pct
0,Carting,3425,3143,0.9177,174928.0000,28.0000,61.0598,26.6181,91.7700
1,FTL,1849,1815,0.9816,388886.0000,84.0000,149.8918,56.6425,98.1600


In [31]:
import os

os.makedirs("outputs/tables", exist_ok=True)

impact_summary.to_csv("outputs/tables/business_impact_scenarios.csv", index=False)
risk_impact_summary.to_csv("outputs/tables/risk_impact_summary.csv", index=False)
recommendation_impact_summary.to_csv("outputs/tables/recommendation_impact_summary.csv", index=False)

print("Saved business impact tables.")

Saved business impact tables.
